# 18 · Qwen3-Reranker-4B + Qwen3-Embedding-8B fusion on the gold standard (post-hoc)

**Post-hoc analysis.** Not pre-registered. It was added after notebook 17 had shown that Qwen3-Reranker-4B scored alone is significantly worse than Qwen3-Embedding-8B on the human expertise labels (0.268 vs 0.222, Holm p = 9.4e-03). On the citation benchmark (notebook 12) the reranker's gain came only from fusion with the embedding score; scored alone it only matched the first stage. The open question is whether the fused score beats the embedding alone on human labels.

**Design (copied, not chosen here).** Fusion form and parameter search are the ones pre-registered for coupling in notebook 14:
- Score = α · z(reranker) + (1 − α) · z(Qwen3-Embedding), z-scores within each participant over their rated papers (`bc.fuse_linear`). **α is the reranker's weight**, the same orientation as notebook 12's α = 0.8 (which used min-max scaling within each list).
- α from the grid 0, 0.1, …, 1, chosen **leave-one-participant-out** (`bc.lopo_select`): each participant's predictions come from the α that minimises the mean pooled loss over the other 57. Only these held-out predictions are tested.
- Inputs are the cached predictions from notebooks 14 and 17 (`predictions/qwen_d_20_*_ta.json`, `predictions/qwen3-reranker_d_20_*_ta.json`), profile samples d_20_1..10. No model is called.
- Pooled loss with the 95% participant bootstrap (`bc.bootstrap_ci`, 1,000 pools, seed 0). **One test**: paired Wilcoxon of per-participant loss, fusion vs Qwen3-Embedding alone. A family of one, so the raw p is reported and no Holm correction applies.
- Descriptive only (no test): fusion at a fixed α = 0.8 (notebook 12's value, transferred) and the in-sample loss for every α.


In [1]:
GOLDSTANDARD_DIR = ""   # absolute path to the gold-standard clone; papermill -p, or env GOLDSTANDARD_DIR
N_BOOT = 1000
SEED = 0

In [2]:
# Parameters
GOLDSTANDARD_DIR = "/Users/senaoz/Developer/SWE-599/research/data/external/goldstandard"


In [3]:
import json
import os
import sys
from collections import Counter
from pathlib import Path

RESEARCH = Path.cwd().parent if Path.cwd().name == "v2" else Path.cwd()
sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

pd.set_option("display.width", 200)

from src import bibcoupling as bc
from src import goldstandard as gs

G = Path(GOLDSTANDARD_DIR or os.environ["GOLDSTANDARD_DIR"])
OUT = RESEARCH / "v2" / "goldstandard"
PRED_DIR = OUT / "predictions"
SAMPLES = list(gs.SAMPLES)

refs = gs.load(G)["refs"]
REVIEWERS = sorted(refs)
print(f"participants {len(REVIEWERS)}, ratings {sum(map(len, refs.values()))}")

participants 58, ratings 477


## Inputs

In [4]:
preds = {m: [json.load(open(PRED_DIR / f"{f}_d_20_{i}_ta.json")) for i in SAMPLES]
         for m, f in (("qwen", "qwen"), ("qwen3_reranker", "qwen3-reranker"))}
for i in range(len(SAMPLES)):
    for rv in REVIEWERS:
        assert set(preds["qwen"][i][rv]) == set(preds["qwen3_reranker"][i][rv]) == set(refs[rv]), (i, rv)

# Both inputs must reproduce the notebook 17 numbers before anything is fused.
check = {m: bc.bootstrap_ci(P, refs, N_BOOT, SEED) for m, P in preds.items()}
for m, (point, lo, hi) in check.items():
    print(f"{m}: {point:.3f} [{lo:.3f}, {hi:.3f}]")
assert abs(check["qwen"][0] - 0.222) < 5e-4 and abs(check["qwen3_reranker"][0] - 0.268) < 5e-4

qwen: 0.222 [0.177, 0.274]
qwen3_reranker: 0.268 [0.210, 0.333]


## Fusion (leave-one-participant-out)

In [5]:
ALPHAS = [round(float(a), 1) for a in np.arange(0, 1.01, 0.1)]
fused = lambda a: [bc.fuse_linear(r, q, a) for r, q in zip(preds["qwen3_reranker"], preds["qwen"])]

preds["fusion_lopo"], choice = bc.lopo_select(fused, ALPHAS, refs)
preds["fusion_alpha_0.8"] = fused(0.8)
print("held-out α choices (α = reranker weight):", dict(sorted(Counter(choice.values()).items())))

insample = pd.Series({a: np.mean([bc.kt_loss(p, refs) for p in fused(a)]) for a in ALPHAS}, name="in-sample pooled loss")
print("participants given α = 0.2 vs 0 (held out) and whether 0.2 helps them in-sample:")
helps = {rv: np.mean([bc.per_reviewer_loss(p, refs)[rv] for p in fused(0.2)]) - np.mean([bc.per_reviewer_loss(p, refs)[rv] for p in fused(0.0)])
         for rv in REVIEWERS if rv in bc.per_reviewer_loss(preds["qwen"][0], refs)}
print(pd.crosstab(pd.Series(choice).loc[list(helps)].rename("held-out α"),
                  pd.Series(helps).map(lambda d: "0.2 better" if d < 0 else "0.2 worse" if d > 0 else "same").rename("in-sample, 0.2 vs 0")))
print("in-sample loss by α (descriptive; α = 0 is Qwen3 alone, α = 1 the reranker alone):")
insample.round(4).to_frame().T

held-out α choices (α = reranker weight): {0.0: 40, 0.2: 18}


participants given α = 0.2 vs 0 (held out) and whether 0.2 helps them in-sample:


in-sample, 0.2 vs 0  0.2 better  0.2 worse  same
held-out α                                      
0.0                          16          0    24
0.2                           0         18     0
in-sample loss by α (descriptive; α = 0 is Qwen3 alone, α = 1 the reranker alone):


,0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0
in-sample pooled loss,0.2223,0.2228,0.2223,0.2262,0.2296,0.2371,0.2418,0.2441,0.2492,0.2553,0.2683


## Results

In [6]:
NAMES = {"qwen": "Qwen3-Embedding-8B (max)", "qwen3_reranker": "Qwen3-Reranker-4B (max)",
         "fusion_lopo": "Reranker + Qwen3, linear z-score, α LOPO [tested]",
         "fusion_alpha_0.8": "Reranker + Qwen3, linear z-score, α = 0.8 [descriptive]"}
rows = []
for m in NAMES:
    point, lo, hi = check[m] if m in check else bc.bootstrap_ci(preds[m], refs, N_BOOT, SEED)
    rows.append({"method": m, "label": NAMES[m], "loss": point, "lo": lo, "hi": hi})
results = pd.DataFrame(rows).set_index("method")
results.round(3)

,label,loss,lo,hi
method,,,,
qwen,Qwen3-Embedding-8B (max),0.222,0.177,0.274
qwen3_reranker,Qwen3-Reranker-4B (max),0.268,0.210,0.333
fusion_lopo,"Reranker + Qwen3, linear z-score, α LOPO [tested]",0.233,0.187,0.289
fusion_alpha_0.8,"Reranker + Qwen3, linear z-score, α = 0.8 [des...",0.249,0.193,0.317


## Paired test (per-participant loss, Wilcoxon; one test, no Holm)

In [7]:
def per_participant(P):
    per = [bc.per_reviewer_loss(p, refs) for p in P]
    return pd.Series({rv: np.mean([d[rv] for d in per]) for rv in per[0]})

pp = {m: per_participant(P) for m, P in preds.items()}
a, b = "fusion_lopo", "qwen"
diff = pp[a] - pp[b]
stat, p = wilcoxon(pp[a], pp[b]) if (diff != 0).any() else (np.nan, 1.0)
tests = pd.DataFrame([{"a": a, "b": b, "participants": len(diff), "mean loss a": pp[a].mean(), "mean loss b": pp[b].mean(),
                       "mean diff (a-b)": diff.mean(), "a better": int((diff < 0).sum()), "b better": int((diff > 0).sum()),
                       "ties": int((diff == 0).sum()), "W": stat, "p": p}])
show = tests.copy()
num = [c for c in show.columns if c.startswith(("mean", "W"))]
show[num] = show[num].round(3)
show["p"] = show["p"].map(lambda x: f"{x:.2e}")   # p-values never rounded to 0
show

,a,b,participants,mean loss a,mean loss b,mean diff (a-b),a better,b better,ties,W,p
0,fusion_lopo,qwen,58,0.269,0.256,0.014,0,18,40,0.0,1.96e-04


## Outputs

In [8]:
for name in ("fusion_lopo",):
    for i, P in zip(SAMPLES, preds[name]):
        with open(PRED_DIR / f"fusion-linear-qwen3-reranker_d_20_{i}_ta.json", "w") as f:
            json.dump(P, f)
results.reset_index().to_csv(OUT / "18_results.csv", index=False, float_format="%.6g")
tests.to_csv(OUT / "18_tests.csv", index=False, float_format="%.6g")
insample.rename_axis("alpha").reset_index().to_csv(OUT / "18_alpha_insample.csv", index=False, float_format="%.6g")
print("written:", sorted(p.name for p in OUT.glob("18_*.csv")))

written: ['18_alpha_insample.csv', '18_results.csv', '18_tests.csv']


### Findings

- **Post-hoc, one test.** Linear z-score fusion of Qwen3-Reranker-4B with Qwen3-Embedding-8B, α (the reranker's weight) chosen leave-one-participant-out on 0, 0.1, …, 1, as notebook 14 did for coupling.
- **Fusion does not beat the embedding on human labels.** Held-out fusion is 0.233 [0.187, 0.289] against Qwen3-Embedding alone at 0.222 [0.177, 0.274]. Per participant it is worse for 18, better for none, and tied for 40 (Wilcoxon on the 18 non-zero differences, p = 1.96e-04, mean +0.014). The reranker alone scores 0.268 [0.210, 0.333].
- **The significant "worse" is mostly a LOPO artefact; the substantive result is "no gain".** The in-sample curve is flat between α = 0 and 0.2 (pooled loss 0.2223 for both) and rises from there to 0.268 at α = 1. Leaving out a participant who gains from α = 0.2 tips the choice to 0, and leaving out one who loses tips it to 0.2. So all 18 participants given α = 0.2 are ones it hurts in-sample, and the 16 it helps all got α = 0. Even the best in-sample α ties the embedding alone.
- **The citation-benchmark setting does not transfer.** Fixed α = 0.8 (notebook 12's value, descriptive) gives 0.249 [0.193, 0.317], between the two inputs.
- **Reading.** On the citation benchmark the reranker helped only when fused (R-precision +3.2, notebook 12). On human expertise labels it adds nothing even when fused, so the label-dependent claim from notebook 17 still holds.
- **Limits.** As for notebooks 14 and 17. The reranker instruction was written for this task without tuning; fusion here uses within-participant z-scores, notebook 12 used min-max scaling within each list.
